# BrainBoost Iteration 3: Zenodo SSAQS Dataset ETL

**Dataset:** A Dataset of University Students' Stress and Anxiety Levels based on Questionnaires and Wearable Sensors  
**Source:** https://zenodo.org/records/18706837  
**DOI:** 10.5281/zenodo.18706837  
**License:** Creative Commons Attribution 4.0 International  

This notebook shows an ETL workflow for an external open dataset that supports BrainBoost Iteration 3. The goal is to transform university-student wearable and questionnaire data into a daily analytical table that can support BrainBoost's planned Smart Reminder and Wearable Health Integration features.

## Why This Dataset Supports BrainBoost

BrainBoost Iteration 3 focuses on Smart Reminder and Wearable Health Integration. This Zenodo dataset is suitable because it contains:

- university student participants
- daily stress and anxiety self-reports
- Fitbit-derived sleep score, deep sleep, stress score, steps, activity level, HRV, and oxygen data
- timestamped records that can be aggregated into daily behavioural features

The dataset was collected outside Australia, so it is used as supporting evidence for prototype logic, not as direct proof of Australian student behaviour.

## ETL Overview

| Stage | Action | Output |
|---|---|---|
| Extract | Read raw Zenodo CSV files from participant folders | Raw sleep, steps, activity, HRV, oxygen, stress, and questionnaire records |
| Transform | Aggregate timestamp-level data into daily records per participant | Daily sleep, movement, stress, anxiety, and wearable summaries |
| Load | Save a BrainBoost-style processed CSV | `data/processed/zenodo_ssaqs_daily_brainboost.csv` |

The ETL script is stored at `scripts/etl_zenodo_ssaqs.py` so the workflow is repeatable.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter

RAW_ROOT = Path('data/external/zenodo_ssaqs/extracted')
ZIP_PATH = Path('data/external/zenodo_ssaqs/SSAQS_dataset.zip')
PROCESSED_CSV = Path('data/processed/zenodo_ssaqs_daily_brainboost.csv')
SUMMARY_JSON = Path('data/processed/zenodo_ssaqs_etl_summary.json')

print('Zip exists:', ZIP_PATH.exists(), ZIP_PATH)
print('Extracted folder exists:', RAW_ROOT.exists(), RAW_ROOT)
print('Processed CSV exists:', PROCESSED_CSV.exists(), PROCESSED_CSV)

## 1. Extract

The extracted Zenodo dataset has root files and participant folders. Root files link students to courses, while participant folders contain wearable and daily questionnaire files.

In [ ]:
participant_dirs = sorted([p for p in RAW_ROOT.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name))
root_files = sorted([p.name for p in RAW_ROOT.iterdir() if p.is_file()])

print('Root files:', root_files)
print('Participant folders:', len(participant_dirs))
print('First five participant folders:', [p.name for p in participant_dirs[:5]])

In [ ]:
csv_counts = Counter()
row_counts = Counter()

for csv_path in RAW_ROOT.rglob('*.csv'):
    csv_counts[csv_path.name] += 1
    with csv_path.open(newline='', encoding='utf-8-sig') as f:
        rows = max(sum(1 for _ in f) - 1, 0)
    row_counts[csv_path.name] += rows

for name in sorted(csv_counts):
    print(f'{name:22s} files={csv_counts[name]:>2} rows={row_counts[name]:>8}')

## Raw File Meaning

| File | Meaning | BrainBoost Relevance |
|---|---|---|
| `daily_questions.csv` | Self-reported stress and anxiety | Smart Reminder risk signals |
| `sleep.csv` | Fitbit sleep overall score and deep sleep | Wearable sleep integration |
| `steps.csv` | Timestamped step records | Movement/activity tracking |
| `activity_level.csv` | Sedentary/light/fair/very active labels | Automatic physical activity signal |
| `hrv.csv` | HRV features such as RMSSD | Stress and recovery indicator |
| `oxygen.csv` | Oxygen measurements | Wearable physiological signal |
| `stress.csv` | Fitbit stress score | Wearable stress feature |
| `users-courses.csv` | Participant to course mapping | Academic context |
| `course-details.csv` | Course schedule information | Study period/context support |

## 2. Transform

The raw files have different granularities. For BrainBoost, daily aggregation is the most useful level because the web app's Habit Tracker stores one check-in per user per day.

The ETL script creates these daily features:

- `sleep_overall_score`
- `deep_sleep_minutes`
- `steps_total`
- `active_minutes`
- `sedentary_minutes`
- `avg_hrv_rmssd`
- `avg_oxygen`
- `fitbit_stress_score`
- `self_report_stress`
- `self_report_anxiety`
- BrainBoost-compatible mapped fields

In [ ]:
# Re-run the repeatable ETL script.
# This reads the extracted Zenodo files and regenerates the processed CSV.
import runpy
runpy.run_path('scripts/etl_zenodo_ssaqs.py', run_name='__main__')

## 3. Load

The processed output is loaded from `data/processed/zenodo_ssaqs_daily_brainboost.csv`. This is the dataset that can be referenced in the report and reused for charts or analysis.

In [ ]:
with SUMMARY_JSON.open(encoding='utf-8') as f:
    summary = json.load(f)

summary

In [ ]:
with PROCESSED_CSV.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    processed_rows = list(reader)

print('Processed daily rows:', len(processed_rows))
print('Columns:', reader.fieldnames)
processed_rows[:3]

## BrainBoost Mapping

The ETL output maps the external wearable dataset into BrainBoost-style fields:

| ETL Field | BrainBoost Use |
|---|---|
| `brainboost_sleep_hours_band` | Comparable to backend `sleep_hours` bands |
| `brainboost_screen_time_proxy` | A proxy risk band, because this dataset has no screen-time field |
| `brainboost_physical_activity` | Comparable to backend `physical_activity` boolean |
| `smart_reminder_candidate` | Whether the daily pattern could justify a reminder |
| `self_report_stress`, `self_report_anxiety` | Smart Reminder context for high-pressure days |
| `steps_total`, `active_minutes` | Wearable movement data for automatic activity tracking |
| `sleep_overall_score`, `deep_sleep_minutes` | Wearable sleep support for dashboard insights |

In [ ]:
def not_empty(value):
    return value not in (None, '')

total_rows = len(processed_rows)
participants = len(set(row['userid'] for row in processed_rows))
reminder_candidates = sum(row['smart_reminder_candidate'] == 'True' for row in processed_rows)
sleep_rows = sum(not_empty(row['sleep_overall_score']) for row in processed_rows)
stress_rows = sum(not_empty(row['self_report_stress']) for row in processed_rows)
activity_true = sum(row['brainboost_physical_activity'] == 'True' for row in processed_rows)

etl_metrics = {
    'daily_rows': total_rows,
    'participants': participants,
    'rows_with_sleep_score': sleep_rows,
    'rows_with_self_report_stress': stress_rows,
    'physically_active_days': activity_true,
    'smart_reminder_candidate_days': reminder_candidates,
    'smart_reminder_candidate_percent': round(reminder_candidates / total_rows * 100, 1) if total_rows else 0,
}

etl_metrics

## Example Reminder Rules

The processed dataset can support Iteration 3 reminder logic such as:

| Condition | Reminder Type |
|---|---|
| Low sleep score | Sleep reset reminder |
| Low steps or low active minutes | Movement break reminder |
| High self-reported stress or anxiety | Study support / wellbeing reminder |
| High Fitbit stress score | Recovery and rest reminder |

This is useful for BrainBoost because it shows how wearable and questionnaire data can be transformed into explainable, student-centred reminder triggers.

In [ ]:
examples = []
for row in processed_rows:
    if row['smart_reminder_candidate'] == 'True':
        reason = []
        if row['sleep_overall_score'] and float(row['sleep_overall_score']) < 70:
            reason.append('low sleep score')
        if row['steps_total'] and int(row['steps_total']) < 5000:
            reason.append('low steps')
        if row['self_report_stress'] and float(row['self_report_stress']) >= 60:
            reason.append('high self-reported stress')
        if row['self_report_anxiety'] and float(row['self_report_anxiety']) >= 60:
            reason.append('high self-reported anxiety')
        examples.append({
            'userid': row['userid'],
            'date': row['date'],
            'sleep_band': row['brainboost_sleep_hours_band'],
            'physical_activity': row['brainboost_physical_activity'],
            'reason': ', '.join(reason) or 'general risk pattern'
        })
    if len(examples) >= 8:
        break

examples

## Report-Ready Summary

The SSAQS dataset was transformed from raw wearable and questionnaire files into a daily BrainBoost-style analytical dataset. The ETL process aggregates timestamp-level Fitbit records into daily features such as sleep score, deep sleep minutes, steps, active minutes, HRV, oxygen, Fitbit stress score, and self-reported stress/anxiety. These fields were then mapped into BrainBoost-compatible concepts such as sleep band, physical activity status, and smart reminder candidate days. This supports Iteration 3 by showing how wearable data can improve data quality and how stress/sleep/activity patterns can drive explainable student-centred reminders.

Limitation: the dataset was collected from university students outside Australia, so it supports technical feasibility and behavioural reasoning but should not be treated as direct evidence of Australian student behaviour.

## 4. Data Quality Assessment

A stronger data science workflow should check whether the ETL output is complete enough to support feature design. This section measures missingness, coverage, and daily record availability.

In [ ]:
columns = reader.fieldnames
missing_summary = []
for col in columns:
    missing = sum(row[col] in ('', None) for row in processed_rows)
    missing_summary.append({
        'column': col,
        'missing_count': missing,
        'missing_percent': round(missing / len(processed_rows) * 100, 1) if processed_rows else 0
    })

missing_summary

In [ ]:
coverage_by_user = {}
for row in processed_rows:
    uid = row['userid']
    coverage_by_user.setdefault(uid, {'days': 0, 'sleep_days': 0, 'stress_days': 0, 'active_days': 0})
    coverage_by_user[uid]['days'] += 1
    if row['sleep_overall_score']:
        coverage_by_user[uid]['sleep_days'] += 1
    if row['self_report_stress']:
        coverage_by_user[uid]['stress_days'] += 1
    if row['brainboost_physical_activity'] == 'True':
        coverage_by_user[uid]['active_days'] += 1

coverage_table = []
for uid, values in coverage_by_user.items():
    coverage_table.append({
        'userid': uid,
        'days': values['days'],
        'sleep_coverage_percent': round(values['sleep_days'] / values['days'] * 100, 1),
        'stress_coverage_percent': round(values['stress_days'] / values['days'] * 100, 1),
        'active_day_percent': round(values['active_days'] / values['days'] * 100, 1),
    })

coverage_table[:10]

## 5. Advanced Feature Engineering

BrainBoost needs explainable features rather than raw sensor streams. The features below translate wearable and questionnaire data into practical behavioural signals.

In [ ]:
def to_float(value):
    try:
        return float(value) if value not in ('', None) else None
    except ValueError:
        return None

def to_int(value):
    try:
        return int(float(value)) if value not in ('', None) else 0
    except ValueError:
        return 0

features = []
for row in processed_rows:
    sleep = to_float(row['sleep_overall_score'])
    stress = to_float(row['self_report_stress'])
    anxiety = to_float(row['self_report_anxiety'])
    fitbit_stress = to_float(row['fitbit_stress_score'])
    steps = to_int(row['steps_total'])
    active_minutes = to_int(row['active_minutes'])
    sedentary_minutes = to_int(row['sedentary_minutes'])
    hrv = to_float(row['avg_hrv_rmssd'])

    low_sleep = sleep is not None and sleep < 70
    high_stress = (stress is not None and stress >= 60) or (fitbit_stress is not None and fitbit_stress >= 75)
    high_anxiety = anxiety is not None and anxiety >= 60
    low_movement = steps < 5000 and active_minutes < 30
    high_sedentary = sedentary_minutes >= 600
    low_recovery = hrv is not None and hrv < 35

    risk_score = (
        low_sleep * 3 +
        high_stress * 3 +
        high_anxiety * 2 +
        low_movement * 2 +
        high_sedentary * 1 +
        low_recovery * 1
    )

    features.append({
        **row,
        'low_sleep_flag': bool(low_sleep),
        'high_stress_flag': bool(high_stress),
        'high_anxiety_flag': bool(high_anxiety),
        'low_movement_flag': bool(low_movement),
        'high_sedentary_flag': bool(high_sedentary),
        'low_recovery_flag': bool(low_recovery),
        'brainboost_risk_score': risk_score,
        'risk_level': 'high' if risk_score >= 6 else 'medium' if risk_score >= 3 else 'low'
    })

features[:3]

## 6. Smart Reminder Decision Engine

This rule engine converts engineered features into reminder categories. It is intentionally explainable because wellbeing-related recommendations should be transparent.

In [ ]:
def assign_reminder(row):
    if row['low_sleep_flag'] and row['high_stress_flag']:
        return 'sleep_and_stress_support', 'Low sleep and high stress detected'
    if row['low_sleep_flag']:
        return 'sleep_reset', 'Wearable sleep score is below the healthy threshold'
    if row['high_stress_flag'] or row['high_anxiety_flag']:
        return 'study_support', 'High stress or anxiety response detected'
    if row['low_movement_flag']:
        return 'movement_break', 'Step count and active minutes are low'
    if row['high_sedentary_flag']:
        return 'sedentary_reset', 'Long sedentary duration detected'
    if row['low_recovery_flag']:
        return 'recovery_check', 'Low HRV recovery signal detected'
    return 'none', 'No strong risk pattern detected'

for row in features:
    reminder_type, reminder_reason = assign_reminder(row)
    row['expert_reminder_type'] = reminder_type
    row['expert_reminder_reason'] = reminder_reason

reminder_counts = Counter(row['expert_reminder_type'] for row in features)
reminder_counts

In [ ]:
expert_examples = [
    {
        'userid': row['userid'],
        'date': row['date'],
        'risk_score': row['brainboost_risk_score'],
        'risk_level': row['risk_level'],
        'reminder_type': row['expert_reminder_type'],
        'reason': row['expert_reminder_reason']
    }
    for row in features
    if row['expert_reminder_type'] != 'none'
][:10]

expert_examples

## 7. Baseline Predictive Framing

A full machine-learning model is not necessary for BrainBoost yet, but a baseline predictive framing helps show data science maturity. Here we define a target called `next_day_high_risk`, which indicates whether the same participant has a high risk score on the following day.

This is not used for medical prediction. It is a prototype framing for evaluating whether today's wearable and questionnaire signals could help anticipate tomorrow's reminder need.

In [ ]:
features_sorted = sorted(features, key=lambda r: (int(r['userid']), r['date']))
by_user = {}
for row in features_sorted:
    by_user.setdefault(row['userid'], []).append(row)

model_rows = []
for uid, rows in by_user.items():
    for current, nxt in zip(rows, rows[1:]):
        model_rows.append({
            'userid': uid,
            'date': current['date'],
            'risk_score_today': current['brainboost_risk_score'],
            'low_sleep_today': current['low_sleep_flag'],
            'high_stress_today': current['high_stress_flag'],
            'low_movement_today': current['low_movement_flag'],
            'next_day_high_risk': nxt['brainboost_risk_score'] >= 6
        })

positive = sum(row['next_day_high_risk'] for row in model_rows)
baseline_rate = round(positive / len(model_rows) * 100, 1) if model_rows else 0
{'model_rows': len(model_rows), 'next_day_high_risk_count': positive, 'baseline_positive_rate_percent': baseline_rate}

## 8. Validation Against BrainBoost Use Cases

| BrainBoost Requirement | Dataset Evidence | ETL Output |
|---|---|---|
| Wearable sleep support | Fitbit sleep and deep sleep files | `sleep_overall_score`, `deep_sleep_minutes`, `brainboost_sleep_hours_band` |
| Movement tracking | Steps and activity level files | `steps_total`, `active_minutes`, `brainboost_physical_activity` |
| Smart Reminder stress context | Daily questions and Fitbit stress files | `self_report_stress`, `self_report_anxiety`, `fitbit_stress_score` |
| Explainable reminder triggers | Rule-based feature flags | `expert_reminder_type`, `expert_reminder_reason` |
| Ethical prototype use | Aggregated daily rows, no raw display | report-ready limitations and citation |

## 9. Expert Data Science Notes

**Why rule-based first:** BrainBoost handles wellbeing-related data. A transparent rule-based reminder engine is safer and easier to explain than an opaque model at prototype stage.

**Why daily aggregation:** The current BrainBoost backend stores one habit record per user per day, so the external dataset is aggregated to the same level.

**Why not direct screen-time mapping:** The SSAQS dataset does not include screen-time data. The `brainboost_screen_time_proxy` field is only a stress-risk proxy for demonstration and should not be described as real screen time.

**Bias limitation:** Participants are Mexican university students using Fitbit Inspire 3 devices. This may not generalise to Australian students or to students without wearable access.

**Privacy limitation:** Even open wearable datasets can be sensitive. BrainBoost should only show aggregated patterns and explainable reminders, not raw participant-level traces.

## Citation

Garcia Ceja, E., Alvarado-Uribe, J., Escamilla-Ambrosio, P. J., Lara, A., Mena-Martinez, A. R., Gallegos-Garcia, G., Gonzalez-Mendoza, M., Monroy, R., Martinez Luna, G. L., & Fernandez-Cardenas, J. M. (2026). *A Dataset of University Students' Stress and Anxiety Levels based on Questionnaires and Wearable Sensors* [Data set]. Zenodo. https://doi.org/10.5281/zenodo.18706837